# R3maJ Kaggle Notebook v4

Kaggle training with Google Drive auth via **service account**, replay/checkpoint restore, automatic checkpoint backup, and a crash-resilient training supervisor (same behavior as the Colab notebook).

Expected Drive layout:
- `serialized_replays.bin`
- `checkpoints/`
- `r3maj_service_account.json`  (service-account key; folder shared with the service account email as Viewer)

The key file is fetched from the folder (or `/kaggle/input` if uploaded as a dataset). No OAuth consent screen, no test users, no refresh tokens.

Run cells top to bottom. Cell 9 starts training; cell 10 stops it cleanly (the binary saves a checkpoint on SIGTERM).

In [ ]:
# 1. Clone R3maJ
import os, subprocess
ROOT='/kaggle/working/R3maJ'
REPO='https://github.com/vfxjamer/R3maJ.git'
if not os.path.isdir(os.path.join(ROOT,'.git')):
    subprocess.run(['git','clone','--depth','1',REPO,ROOT],check=True)
else:
    subprocess.run(['git','-C',ROOT,'pull'],check=False)
print(ROOT)


In [ ]:
# 2. Install dependencies and define paths
!pip install -q -U 'gdown>=6.1.0' google-api-python-client google-auth google-auth-httplib2 google-auth-oauthlib

import os, json, shutil, time, threading, glob
import gdown
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload
from google_auth_oauthlib.flow import InstalledAppFlow
from google.oauth2.credentials import Credentials

DRIVE_ROOT_ID='1ktqAHT6wwYCyyRA4REBN2kZyFqeCbvKh'
DRIVE_FOLDER_URL=f'https://drive.google.com/drive/folders/{DRIVE_ROOT_ID}'
LOCAL_ROOT='/kaggle/working/R3maJ/build'
LOCAL_REPLAY=f'{LOCAL_ROOT}/serialized_replays.bin'
LOCAL_CHECKPOINTS=f'{LOCAL_ROOT}/checkpoints'
CLIENT_SECRET_NAME='client_secret_446177867585-k3dq9qglmq10q09mqg8vt9lf1se3nmfn.apps.googleusercontent.com'
TOKEN_PATH='/kaggle/working/google_drive_token.json'
SCOPES=['https://www.googleapis.com/auth/drive']
os.makedirs(LOCAL_ROOT,exist_ok=True)
print('gdown:',gdown.__version__)
print('Drive root:',DRIVE_ROOT_ID)


In [ ]:
# 3. Service-account authentication (no OAuth consent screen needed)
# Download r3maj_service_account.json from the Drive folder (shared "Anyone with
# the link -> Viewer"), or read it from /kaggle/input if uploaded as a dataset.
SERVICE_ACCOUNT_NAME='r3maj_service_account.json'

def locate_service_account():
    search_roots=['/kaggle/working','/kaggle/input','/kaggle/working/R3maJ']
    for base in search_roots:
        if os.path.exists(base):
            hits=glob.glob(os.path.join(base,'**',SERVICE_ACCOUNT_NAME),recursive=True)
            if hits:
                return hits[0]

    print('Service account not local; resolving it from the public Drive folder...')
    entries=gdown.download_folder(
        url=DRIVE_FOLDER_URL,
        skip_download=True,
        quiet=True
    )
    matches=[e for e in entries if os.path.basename(e.path)==SERVICE_ACCOUNT_NAME]
    if not matches:
        names=[e.path for e in entries]
        raise FileNotFoundError(
            f'{SERVICE_ACCOUNT_NAME} was not found in the Drive folder. '
            f'Make the folder public (Anyone with the link -> Viewer) and keep the filename exact. '
            f'Visible entries: {names[:20]}'
        )

    sa_id=matches[0].id
    dest=f'/kaggle/working/{SERVICE_ACCOUNT_NAME}'
    gdown.download(id=sa_id, output=dest, quiet=False)
    if not os.path.isfile(dest):
        raise FileNotFoundError(f'Failed to download service account JSON: {dest}')
    return dest

sa_path=locate_service_account()
print('Service account key:',sa_path)

from google.oauth2 import service_account
creds=service_account.Credentials.from_service_account_file(
    sa_path, scopes=SCOPES
)
drive=build('drive','v3',credentials=creds,cache_discovery=False)
print('Drive authentication: OK (service account)')


In [ ]:
# 4. Restore serialized_replays.bin + checkpoints from authenticated Drive
def drive_find_children(parent_id,name=None,mime_type=None):
    q=f"'{parent_id}' in parents and trashed = false"
    if name is not None:
        safe=name.replace("'","\\'")
        q += f" and name = '{safe}'"
    if mime_type:
        q += f" and mimeType = '{mime_type}'"
    return drive.files().list(
        q=q,
        fields='files(id,name,mimeType,size,modifiedTime)',
        pageSize=1000
    ).execute().get('files',[])

def drive_download_file(file_id,dest):
    os.makedirs(os.path.dirname(dest),exist_ok=True)
    request=drive.files().get_media(fileId=file_id)
    with open(dest,'wb') as fh:
        downloader=MediaIoBaseDownload(fh,request,chunksize=16*1024*1024)
        done=False
        while not done:
            _,done=downloader.next_chunk()

def drive_download_tree(folder_id,local_dir):
    os.makedirs(local_dir,exist_ok=True)
    for item in drive_find_children(folder_id):
        path=os.path.join(local_dir,item['name'])
        if item['mimeType']=='application/vnd.google-apps.folder':
            drive_download_tree(item['id'],path)
        else:
            drive_download_file(item['id'],path)

DOWNLOAD_DIR='/kaggle/working/R3maJ_drive'
if os.path.exists(DOWNLOAD_DIR):
    shutil.rmtree(DOWNLOAD_DIR)
drive_download_tree(DRIVE_ROOT_ID,DOWNLOAD_DIR)

def find_file(base,name):
    for root,dirs,files in os.walk(base):
        if name in files:
            return os.path.join(root,name)

def find_dir(base,name):
    for root,dirs,files in os.walk(base):
        if name in dirs:
            return os.path.join(root,name)

src_replay=find_file(DOWNLOAD_DIR,'serialized_replays.bin')
src_checkpoints=find_dir(DOWNLOAD_DIR,'checkpoints')
assert src_replay,'serialized_replays.bin not found in Drive folder.'
assert src_checkpoints,'checkpoints folder not found in Drive folder.'

shutil.copy2(src_replay,LOCAL_REPLAY)
if os.path.exists(LOCAL_CHECKPOINTS):
    shutil.rmtree(LOCAL_CHECKPOINTS)
shutil.copytree(src_checkpoints,LOCAL_CHECKPOINTS)

print('Replay:',LOCAL_REPLAY)
print('Replay GB:',round(os.path.getsize(LOCAL_REPLAY)/(1024**3),3))
print('Checkpoint entries:',sorted(os.listdir(LOCAL_CHECKPOINTS))[:20])


In [ ]:
# 5. Install build dependencies + inspect GPU
import subprocess, os
subprocess.run(['apt-get','update','-qq'],check=False)
subprocess.run(['apt-get','install','-y','-qq','build-essential','cmake','git','libpython3-dev','pkg-config'],check=False)

import torch
print('torch:',torch.__version__)
print('CUDA:',torch.cuda.is_available(),torch.version.cuda)
print('GPU count:',torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f'GPU {i}: {torch.cuda.get_device_name(i)}')
assert torch.cuda.is_available(),'CUDA GPU required.'
subprocess.run(['nvidia-smi'],check=False)


In [ ]:
# 6. Configure + build
import os, subprocess, torch
os.chdir(ROOT)
prefix=os.path.dirname(torch.__file__)
subprocess.run([
    'cmake','-S','.','-B','build','-DCMAKE_BUILD_TYPE=Release',
    f'-DTORCH_INSTALL_PREFIX={prefix}'
],check=True)
subprocess.run(['cmake','--build','build','-j',str(os.cpu_count() or 2)],check=True)
EXE=os.path.join(ROOT,'build','R3maJ')
print('Binary:',EXE,os.path.exists(EXE))


In [ ]:
# 7. Verify restored data
assert os.path.exists(EXE),'R3maJ binary missing.'
assert os.path.exists(LOCAL_REPLAY),'Replay missing.'
assert os.path.isdir(LOCAL_CHECKPOINTS),'Checkpoint directory missing.'
print('READY')
TRAIN_GAMES=256
print('Games:',TRAIN_GAMES)
print('Checkpoint entries:',len([x for x in os.listdir(LOCAL_CHECKPOINTS) if not x.startswith('.')]))


In [ ]:
# 8. Checkpoint backup watcher -> GitHub Releases (every 60s)
# Requires a Kaggle secret named GITHUB_PAT (fetched at runtime via
# UserSecretsClient - works even if added after the session started).
# Each new checkpoint dir becomes a release "ckpt-<timestep>" with asset
# "checkpoint_<timestep>.zip". Point GITHUB_REPO at a private repo if needed.
import os, json, glob, shutil, subprocess, time, threading

GITHUB_REPO='vfxjamer/R3maJ'
GITHUB_PAT=''
try:
    from kaggle_secrets import UserSecretsClient
    GITHUB_PAT=UserSecretsClient().get_secret('GITHUB_PAT') or ''
except Exception:
    GITHUB_PAT=os.environ.get('GITHUB_PAT','')
SYNC_INTERVAL=60
_stop_backup=False
_uploaded_dirs=set()

def gh_head():
    return ['-H', f'Authorization: Bearer {GITHUB_PAT}', '-H', 'Accept: application/vnd.github+json']

def gh_release_upload(local_zip, tag, asset_name):
    r=subprocess.run(['curl','-s','-X','POST',
        f'https://api.github.com/repos/{GITHUB_REPO}/releases',
        *gh_head(), '-H', 'Content-Type: application/json',
        '-d', json.dumps({'tag_name': tag, 'name': tag})], capture_output=True, text=True)
    try:
        upl=json.loads(r.stdout)['upload_url'].split('{')[0]
    except (ValueError, KeyError):
        r=subprocess.run(['curl','-s',
            f'https://api.github.com/repos/{GITHUB_REPO}/releases/tags/{tag}',
            *gh_head()], capture_output=True, text=True)
        try:
            rel=json.loads(r.stdout)
            upl=rel['upload_url'].split('{')[0]
        except (ValueError, KeyError):
            return False
        ra=subprocess.run(['curl','-s',
            f'https://api.github.com/repos/{GITHUB_REPO}/releases/{rel["id"]}/assets',
            *gh_head()], capture_output=True, text=True)
        for a in json.loads(ra.stdout):
            if a['name']==asset_name:
                subprocess.run(['curl','-s','-X','DELETE',a['url'],'-H',
                    f'Authorization: Bearer {GITHUB_PAT}'], capture_output=True, text=True)
    r3=subprocess.run(['curl','-s','-X','POST',upl+'?name='+asset_name,
        *gh_head(), '-H','Content-Type: application/zip',
        '--data-binary','@'+local_zip], capture_output=True, text=True)
    try:
        return json.loads(r3.stdout).get('state')=='uploaded'
    except ValueError:
        return False

def backup_new_checkpoints():
    for d in sorted(x for x in glob.glob(os.path.join(LOCAL_CHECKPOINTS,'*'))
                    if os.path.isdir(x)):
        name=os.path.basename(d)
        if name in _uploaded_dirs:
            continue
        tmp=f'/kaggle/working/ckpt_{name}'
        shutil.make_archive(tmp,'zip',d)
        if gh_release_upload(tmp+'.zip','ckpt-'+name,'checkpoint_'+name+'.zip'):
            _uploaded_dirs.add(name)
            print('[backup] uploaded release ckpt-'+name)
        else:
            print('[backup] FAILED to upload', name)

def backup_loop():
    print('[backup] GitHub-Releases watcher started; every', SYNC_INTERVAL, 's',
          '(PAT present)' if GITHUB_PAT else '(NO GITHUB_PAT secret - backups disabled)')
    while not _stop_backup:
        try:
            if GITHUB_PAT:
                backup_new_checkpoints()
        except Exception as e:
            print('[backup] ERROR:', repr(e))
        time.sleep(SYNC_INTERVAL)

threading.Thread(target=backup_loop,daemon=True).start()


In [ ]:
# 9. Training supervisor (mirrors Colab: streams logs, auto-restarts on crash, SIGTERM stop)
import os, subprocess, time, threading

os.chdir(os.path.join(ROOT,'build'))
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
REPLAY_ARG=['--replays','serialized_replays.bin'] if os.path.exists('serialized_replays.bin') else []
BASE_CMD=['stdbuf','-oL','-eL','./R3maJ','--device',DEVICE,'--phase','-1',
          '--save-dir','checkpoints','--games',str(TRAIN_GAMES)]+REPLAY_ARG
print('BASE CMD:',' '.join(BASE_CMD))
STOP_FLAG='/kaggle/working/STOP_TRAINING'
if os.path.exists(STOP_FLAG):
    os.remove(STOP_FLAG)

print('[supervisor] training until STOP_TRAINING flag or cell 10 is run')
backoff=5
while not os.path.exists(STOP_FLAG):
    print(f'[supervisor] launching training (backoff {backoff}s)...')
    proc=subprocess.Popen(BASE_CMD,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    try:
        for line in proc.stdout:
            print(line,end='')
    except Exception:
        pass
    if os.path.exists(STOP_FLAG):
        print('[supervisor] SIGTERM (binary saves a checkpoint then exits)')
        proc.terminate()
        try:
            proc.wait(timeout=60)
        except subprocess.TimeoutExpired:
            print('[supervisor] SIGKILL fallback')
            proc.kill()
            proc.wait()
        break
    code=proc.wait()
    print(f'[supervisor] training exited rc={code}; restarting in {backoff}s...')
    time.sleep(backoff)
    if code==0:
        backoff=5
    else:
        backoff=min(backoff*2,60)
print('[supervisor] finished')


In [ ]:
# 10. STOP training cleanly
# The binary saves a checkpoint when it receives SIGTERM, so run this cell to
# stop after a graceful checkpoint. (Falls back to SIGKILL if it hangs.)
import os
open('/kaggle/working/STOP_TRAINING','w').close()
print('STOP_TRAINING flag set - supervisor will terminate the binary cleanly.')
